# 5. The hierarchy of missions, and what each field explains

At coarse resolution the diffusion links neighbouring missions and the clusters merge. On Top-1 this happens in a fixed
order, which the fields explain.

In [ ]:
import sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "src"), str(ROOT)]
import numpy as np
import matplotlib.pyplot as plt
import figlib as L
L.style()
PAPER = ROOT / "results" / "epuck" / "paper"
RES = ROOT / "results" / "epuck"
load = lambda p: pickle.load(open(p, "rb"))

In [ ]:
import make_figures as M
from IPython.display import Image, display
M.fig_fusion_sequence()
display(Image(filename=str(ROOT / "figures/fig_fusion_sequence.png"), width=950))

**What it shows.** No setting of the main grid joins two missions. For every neighbourhood large enough to merge within
the extended grid, the first merge is **D-Gate + Foraging**, one mission absorbed whole (the merged cluster holds exactly
the runs of the two missions, so the purity loses exactly 50 runs), and the second **Shelter + XOR Aggregation**; Homing
stays apart over the whole extended grid.
The next cell prints, for each neighbourhood, every change of the merged missions with its purity. The count never increases with $t$ at fixed $k_{nn}$.

In [ ]:
E = load(PAPER / "extended_top1.pkl")
y = E["labels"]
names = ["D-Gate", "Foraging", "Homing", "Shelter", "XOR"]


def merged(pred):
    """the missions whose runs fall mostly in one same cluster"""
    home = {}
    for m in np.unique(y):
        home.setdefault(int(np.bincount(pred[y == m]).argmax()), []).append(names[m])
    return [" + ".join(g) for g in home.values() if len(g) > 1]


for k in E["knns"]:                       # each time the merged missions change, as t grows
    last = None
    for r in sorted((r for r in E["results"] if r["knn"] == k), key=lambda r: r["t"]):
        g = merged(r["pred"])
        if g and g != last:
            print(f"k_nn={k:2d}, t={r['t']:3d}: C={r['C']}, purity {r['purity']:.3f}, merged: {'; '.join(g)}")
            last = g

## Which field separates which pair?

Separation ratio = mean distance between two missions / mean distance within them (rank distances, averaged over the
scales). Bold = best separator of the pair.

In [ ]:
R, pairs = M.fig_pair_separation()
display(Image(filename=str(ROOT / "figures/fig_pair_separation.png"), width=950))

* **D-Gate / Foraging**: the closest pair in density, kinetic intensity, divergence potential and coupling (continuous
  traffic across the arena), separated only by the **vorticity potential** $\psi$: they differ in how the traffic turns.
  They merge first.
* **Shelter / XOR**: both end with the robots gathered on a patch. They are less separated than the median pair in every
  field but the **coupling** $J$, their best separator, which records where the flow converges relative to where it
  turns. They merge second.
* **Homing**: a mixed signature. By its density and its coupling it resembles the aggregating missions (it ends gathered
  on a spot), by its divergence potential it is closest to XOR Aggregation, and by its kinetic intensity it resembles the
  moving missions (its robots cross the whole arena first). It stays apart over the whole extended grid.

In [ ]:
E = load(PAPER / "extended_top1.pkl")
from sklearn.metrics.cluster import contingency_matrix
yy = E["labels"]
for r in sorted(E["results"], key=lambda r: (r["knn"], r["t"])):
    if r["C"] == 2:
        print(f"C=2 at k_nn={r['knn']}, t={r['t']}:"); print(contingency_matrix(yy, r["pred"]))